# Hidden Markov Model for Human Activity Recognition

I built this notebook to recognize four activities (standing, walking, jumping, still) from accelerometer and gyroscope data using a Hidden Markov Model.

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.fft import fft
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score

# I set a random seed so my results stay the same each time I run the notebook
np.random.seed(42)

ModuleNotFoundError: No module named 'seaborn'

## 2. Data Collection and Loading

I recorded data using the Sensor Logger app on my phone at about **100 Hz** sampling rate. Each recording is 5-10 seconds long.

In [ ]:
# I define my four activities and map them to numbers for the HMM
ACTIVITIES = ['standing', 'walking', 'jumping', 'still']
STATE_NAMES = {0: 'standing', 1: 'walking', 2: 'jumping', 3: 'still'}
ACTIVITY_TO_STATE = {name: i for i, name in enumerate(ACTIVITIES)}

# My sampling rate is about 100 Hz (one sample every 0.01 seconds)
SAMPLING_RATE = 100

# I use 1 second windows because at 100 Hz I get 100 samples per window
# This gives me enough data points for FFT and stable statistics
WINDOW_SIZE = SAMPLING_RATE
WINDOW_STEP = SAMPLING_RATE // 2  # I move the window by 0.5 seconds (50% overlap)

In [ ]:
def load_sensor_pair(activity_folder, sample_number):
  """I load the accelerometer and gyroscope CSV files for one recording."""
  accel_path = os.path.join(activity_folder, f"{sample_number}_accel.csv")
  gyro_path = os.path.join(activity_folder, f"{sample_number}_gyro.csv")
  
  accel = pd.read_csv(accel_path)
  gyro = pd.read_csv(gyro_path)
  
  return accel, gyro


def get_sample_ids(folder, activity_name):
  """I find all sample numbers in a folder (e.g. standing_01, standing_02)."""
  files = os.listdir(folder)
  ids = []
  for f in files:
    if f.endswith('_accel.csv'):
      # I extract the sample id like 'standing_01' from 'standing_01_accel.csv'
      sample_id = f.replace('_accel.csv', '')
      ids.append(sample_id)
  return sorted(ids)

In [ ]:
# I load one example file to check my data
example_accel, example_gyro = load_sensor_pair('walking', 'walking_01')

duration = example_accel['seconds_elapsed'].max() - example_accel['seconds_elapsed'].min()
median_dt = example_accel['seconds_elapsed'].diff().median()
estimated_rate = 1.0 / median_dt

print(f"Number of rows: {len(example_accel)}")
print(f"Duration: {duration:.2f} seconds")
print(f"Estimated sampling rate: {estimated_rate:.1f} Hz")
print(f"Columns: {list(example_accel.columns)}")

## 3. Visualize Sample Sensor Data

In [ ]:
def plot_raw_sensor_data(accel, gyro, title):
  """I plot accelerometer and gyroscope signals to see what my data looks like."""
  fig, axes = plt.subplots(2, 1, figsize=(12, 6))
  
  time = accel['seconds_elapsed']
  
  # I plot accelerometer x, y, z axes
  axes[0].plot(time, accel['x'], label='x', alpha=0.8)
  axes[0].plot(time, accel['y'], label='y', alpha=0.8)
  axes[0].plot(time, accel['z'], label='z', alpha=0.8)
  axes[0].set_title(f'{title} - Accelerometer')
  axes[0].set_xlabel('Time (seconds)')
  axes[0].set_ylabel('Acceleration')
  axes[0].legend()
  axes[0].grid(True, alpha=0.3)
  
  # I plot gyroscope x, y, z axes
  axes[1].plot(time, gyro['x'], label='x', alpha=0.8)
  axes[1].plot(time, gyro['y'], label='y', alpha=0.8)
  axes[1].plot(time, gyro['z'], label='z', alpha=0.8)
  axes[1].set_title(f'{title} - Gyroscope')
  axes[1].set_xlabel('Time (seconds)')
  axes[1].set_ylabel('Angular velocity')
  axes[1].legend()
  axes[1].grid(True, alpha=0.3)
  
  plt.tight_layout()
  plt.show()

In [ ]:
# I plot one sample from each activity to compare the signals
for activity in ACTIVITIES:
  sample_id = f"{activity}_01"
  accel, gyro = load_sensor_pair(activity, sample_id)
  plot_raw_sensor_data(accel, gyro, activity.capitalize())

## 4. Feature Extraction

I extract time-domain and frequency-domain features from each window of sensor data.

In [ ]:
def compute_magnitude(df):
  """I compute the signal magnitude from x, y, z axes."""
  return np.sqrt(df['x']**2 + df['y']**2 + df['z']**2)


def extract_time_features(accel_window, gyro_window):
  """I extract time-domain features from one window of data."""
  accel_mag = compute_magnitude(accel_window).values
  gyro_mag = compute_magnitude(gyro_window).values
  
  features = []
  
  # Mean - I use this because different activities have different average motion levels
  features.append(np.mean(accel_mag))
  
  # Variance - jumping and walking have higher variance than still
  features.append(np.var(accel_mag))
  
  # Standard deviation - similar to variance, helps separate active vs inactive
  features.append(np.std(accel_mag))
  
  # Signal Magnitude Area (SMA) - sum of absolute values of all axes
  sma = np.mean(np.abs(accel_window['x']) + np.abs(accel_window['y']) + np.abs(accel_window['z']))
  features.append(sma)
  
  # Correlation between x and y axes - captures how axes move together
  corr_xy = np.corrcoef(accel_window['x'], accel_window['y'])[0, 1]
  if np.isnan(corr_xy):
    corr_xy = 0.0
  features.append(corr_xy)
  
  # Gyroscope mean - helps distinguish rotating motions
  features.append(np.mean(gyro_mag))
  
  return np.array(features)


def extract_frequency_features(accel_window, sampling_rate):
  """I extract frequency-domain features using FFT."""
  accel_mag = compute_magnitude(accel_window).values
  
  # I apply FFT to find frequency content
  n = len(accel_mag)
  fft_values = fft(accel_mag)
  freqs = np.fft.fftfreq(n, d=1.0/sampling_rate)
  
  # I only use positive frequencies
  positive_mask = freqs > 0
  freqs_positive = freqs[positive_mask]
  power = np.abs(fft_values[positive_mask]) ** 2
  
  features = []
  
  # Dominant frequency - walking has a clear step frequency
  dominant_freq = freqs_positive[np.argmax(power)]
  features.append(dominant_freq)
  
  # Spectral energy - total energy in the frequency domain
  spectral_energy = np.sum(power)
  features.append(spectral_energy)
  
  return np.array(features)

In [ ]:
def extract_features_from_recording(accel, gyro, sampling_rate, window_size, window_step):
  """I split a recording into windows and extract features from each window."""
  all_features = []
  n_samples = len(accel)
  
  start = 0
  while start + window_size <= n_samples:
    accel_window = accel.iloc[start:start + window_size]
    gyro_window = gyro.iloc[start:start + window_size]
    
    time_feats = extract_time_features(accel_window, gyro_window)
    freq_feats = extract_frequency_features(accel_window, sampling_rate)
    
    combined = np.concatenate([time_feats, freq_feats])
    all_features.append(combined)
    
    start += window_step
  
  return np.array(all_features)


FEATURE_NAMES = [
  'accel_mean', 'accel_variance', 'accel_std', 'sma', 'corr_xy', 'gyro_mean',
  'dominant_freq', 'spectral_energy'
]

## 5. Build Training and Test Datasets

I use samples 01-08 for training and samples 09-10 as unseen test data.

In [ ]:
def load_dataset(data_root, sample_filter=None):
  """I load all recordings and extract features with their activity labels."""
  sequences = []
  labels = []
  
  for activity in ACTIVITIES:
    folder = os.path.join(data_root, activity)
    sample_ids = get_sample_ids(folder, activity)
    
    for sample_id in sample_ids:
      # I check if this sample should be included based on the filter
      sample_num = sample_id.split('_')[-1]
      if sample_filter is not None:
        if sample_num not in sample_filter:
          continue
      
      accel, gyro = load_sensor_pair(folder, sample_id)
      features = extract_features_from_recording(
        accel, gyro, SAMPLING_RATE, WINDOW_SIZE, WINDOW_STEP
      )
      
      if len(features) > 0:
        sequences.append(features)
        state = ACTIVITY_TO_STATE[activity]
        labels.append(np.full(len(features), state))
  
  return sequences, labels


# Training data: samples 01 to 08
train_sequences, train_labels = load_dataset('.', sample_filter=[f'{i:02d}' for i in range(1, 9)])

# Test data: samples 09 and 10 (unseen data)
test_sequences, test_labels = load_dataset('test_data', sample_filter=None)

print(f"Training sequences: {len(train_sequences)}")
print(f"Test sequences: {len(test_sequences)}")
print(f"Features per window: {train_sequences[0].shape[1]}")
print(f"Windows in first training sequence: {len(train_sequences[0])}")

In [ ]:
# I normalize features using Z-score so all features have similar scale
all_train_features = np.vstack(train_sequences)
scaler = StandardScaler()
scaler.fit(all_train_features)

# I apply normalization to training and test data
train_sequences_scaled = [scaler.transform(seq) for seq in train_sequences]
test_sequences_scaled = [scaler.transform(seq) for seq in test_sequences]

print(f"Total training windows: {sum(len(s) for s in train_sequences_scaled)}")
print(f"Total test windows: {sum(len(s) for s in test_sequences_scaled)}")

## 6. Hidden Markov Model Implementation

I implement the HMM from scratch with Gaussian emissions, Viterbi decoding, and Baum-Welch training.

In [ ]:
class GaussianHMM:
  """My Hidden Markov Model with Gaussian emission probabilities."""
  
  def __init__(self, n_states, n_features):
    self.n_states = n_states
    self.n_features = n_features
    
    # Initial state probabilities (pi)
    self.pi = np.ones(n_states) / n_states
    
    # Transition matrix (A) - I start with high self-transition probability
    self.A = np.ones((n_states, n_states)) * 0.1
    np.fill_diagonal(self.A, 0.7)
    self.A = self.A / self.A.sum(axis=1, keepdims=True)
    
    # Emission parameters: mean and covariance for each state
    self.means = np.random.randn(n_states, n_features) * 0.1
    self.covs = np.array([np.eye(n_features) for _ in range(n_states)])
  
  def _gaussian_log_prob(self, x, mean, cov):
    """I compute the log probability of observation x under a Gaussian distribution."""
    d = len(x)
    diff = x - mean
    
    # I add a small value to the diagonal for numerical stability
    cov_stable = cov + np.eye(d) * 1e-6
    
    try:
      inv_cov = np.linalg.inv(cov_stable)
      sign, log_det = np.linalg.slogdet(cov_stable)
    except np.linalg.LinAlgError:
      return -1e10
    
    log_prob = -0.5 * (d * np.log(2 * np.pi) + log_det + diff @ inv_cov @ diff)
    return log_prob
  
  def _emission_matrix(self, observations):
    """I compute the emission probability for each state at each time step."""
    T = len(observations)
    B = np.zeros((T, self.n_states))
    
    for t in range(T):
      for s in range(self.n_states):
        B[t, s] = self._gaussian_log_prob(observations[t], self.means[s], self.covs[s])
    
    return B
  
  def forward(self, observations):
    """I run the forward algorithm to compute alpha values."""
    T = len(observations)
    B = self._emission_matrix(observations)
    
    log_alpha = np.zeros((T, self.n_states))
    
    # Initialization
    log_alpha[0] = np.log(self.pi + 1e-300) + B[0]
    
    for t in range(1, T):
      for s in range(self.n_states):
        log_alpha[t, s] = B[t, s] + self._log_sum_exp(
          log_alpha[t-1] + np.log(self.A[:, s] + 1e-300)
        )
    
    log_likelihood = self._log_sum_exp(log_alpha[-1])
    return log_alpha, log_likelihood
  
  def backward(self, observations):
    """I run the backward algorithm to compute beta values."""
    T = len(observations)
    B = self._emission_matrix(observations)
    
    log_beta = np.zeros((T, self.n_states))
    
    for t in range(T - 2, -1, -1):
      for s in range(self.n_states):
        log_beta[t, s] = self._log_sum_exp(
          np.log(self.A[s] + 1e-300) + B[t+1] + log_beta[t+1]
        )
    
    return log_beta
  
  def _log_sum_exp(self, log_values):
    """I compute log(sum(exp(x))) in a numerically stable way."""
    max_val = np.max(log_values)
    return max_val + np.log(np.sum(np.exp(log_values - max_val)))
  
  def viterbi(self, observations):
    """I use the Viterbi algorithm to find the most likely state sequence."""
    T = len(observations)
    B = self._emission_matrix(observations)
    
    # V stores log probabilities, path stores backpointers
    V = np.zeros((T, self.n_states))
    path = np.zeros((T, self.n_states), dtype=int)
    
    # Initialization
    V[0] = np.log(self.pi + 1e-300) + B[0]
    
    # Recursion
    for t in range(1, T):
      for s in range(self.n_states):
        probs = V[t-1] + np.log(self.A[:, s] + 1e-300)
        path[t, s] = np.argmax(probs)
        V[t, s] = probs[path[t, s]] + B[t, s]
    
    # Backtracking
    states = np.zeros(T, dtype=int)
    states[-1] = np.argmax(V[-1])
    
    for t in range(T - 2, -1, -1):
      states[t] = path[t + 1, states[t + 1]]
    
    return states
  
  def baum_welch(self, sequences, max_iter=50, epsilon=1e-4):
    """I train the HMM using the Baum-Welch algorithm."""
    prev_log_likelihood = -np.inf
    
    for iteration in range(max_iter):
      # I accumulate statistics across all sequences
      pi_num = np.zeros(self.n_states)
      A_num = np.zeros((self.n_states, self.n_states))
      A_den = np.zeros(self.n_states)
      mean_num = np.zeros((self.n_states, self.n_features))
      cov_num = np.zeros((self.n_states, self.n_features, self.n_features))
      state_den = np.zeros(self.n_states)
      
      total_log_likelihood = 0
      
      for observations in sequences:
        T = len(observations)
        B = self._emission_matrix(observations)
        log_alpha, log_ll = self.forward(observations)
        log_beta = self.backward(observations)
        total_log_likelihood += log_ll
        
        # I compute gamma (state occupancy probabilities)
        log_gamma = log_alpha + log_beta
        
        # I normalize gamma at each time step
        for t in range(T):
          log_gamma[t] -= self._log_sum_exp(log_gamma[t])
        gamma = np.exp(log_gamma)
        
        # I update pi
        pi_num += gamma[0]
        
        # I update means and covariances
        for s in range(self.n_states):
          state_den[s] += np.sum(gamma[:, s])
          mean_num[s] += gamma[:, s] @ observations
          
          for t in range(T):
            diff = observations[t] - self.means[s]
            cov_num[s] += gamma[t, s] * np.outer(diff, diff)
        
        # I compute xi (transition probabilities)
        for t in range(T - 1):
          for i in range(self.n_states):
            for j in range(self.n_states):
              log_xi = (log_alpha[t, i] + np.log(self.A[i, j] + 1e-300) +
                       B[t+1, j] + log_beta[t+1, j])
              norm = self._log_sum_exp(log_alpha[t] + log_beta[t])
              xi = np.exp(log_xi - norm)
              A_num[i, j] += xi
      
      # M-step: I update model parameters
      self.pi = pi_num / len(sequences)
      self.pi = self.pi / self.pi.sum()
      
      for i in range(self.n_states):
        row_sum = A_num[i].sum()
        if row_sum > 0:
          self.A[i] = A_num[i] / row_sum
      
      for s in range(self.n_states):
        if state_den[s] > 0:
          self.means[s] = mean_num[s] / state_den[s]
          self.covs[s] = cov_num[s] / state_den[s]
          # I add regularization to avoid singular matrices
          self.covs[s] += np.eye(self.n_features) * 1e-4
      
      # I check convergence using log-likelihood change
      avg_log_likelihood = total_log_likelihood / len(sequences)
      change = avg_log_likelihood - prev_log_likelihood
      
      print(f"Iteration {iteration + 1}: log-likelihood = {avg_log_likelihood:.4f}, change = {change:.6f}")
      
      if abs(change) < epsilon and iteration > 0:
        print(f"I stopped training because the model converged (change < {epsilon})")
        break
      
      prev_log_likelihood = avg_log_likelihood
    
    return self

In [ ]:
def initialize_hmm_from_labels(sequences, labels, n_states, n_features):
  """I initialize the HMM using labeled training data for better starting values."""
  model = GaussianHMM(n_states, n_features)
  
  all_features = np.vstack(sequences)
  all_labels = np.concatenate(labels)
  
  # I set emission means and covariances from labeled data
  for s in range(n_states):
    state_features = all_features[all_labels == s]
    if len(state_features) > 0:
      model.means[s] = np.mean(state_features, axis=0)
      model.covs[s] = np.cov(state_features.T) + np.eye(n_features) * 1e-4
  
  # I estimate initial probabilities from first window of each sequence
  first_states = [label[0] for label in labels]
  for s in range(n_states):
    model.pi[s] = first_states.count(s) / len(first_states)
  
  # I estimate transition matrix from labeled sequences
  trans_counts = np.ones((n_states, n_states)) * 0.01  # small smoothing
  for label_seq in labels:
    for t in range(len(label_seq) - 1):
      trans_counts[label_seq[t], label_seq[t+1]] += 1
  
  for i in range(n_states):
    model.A[i] = trans_counts[i] / trans_counts[i].sum()
  
  return model

## 7. Train the HMM

In [ ]:
n_states = 4
n_features = len(FEATURE_NAMES)

# I initialize the model using my labeled training data
model = initialize_hmm_from_labels(
  train_sequences_scaled, train_labels, n_states, n_features
)

print("Initial transition matrix:")
print(np.round(model.A, 3))
print()

# I train the model with Baum-Welch
print("Training with Baum-Welch algorithm...")
model.baum_welch(train_sequences_scaled, max_iter=30, epsilon=1e-3)

## 8. Visualize Transition Matrix

In [ ]:
def plot_transition_matrix(A, state_names):
  """I plot the transition matrix as a heatmap."""
  labels = [state_names[i] for i in range(len(state_names))]
  
  plt.figure(figsize=(8, 6))
  sns.heatmap(A, annot=True, fmt='.3f', cmap='Blues',
              xticklabels=labels, yticklabels=labels)
  plt.title('HMM Transition Probability Matrix')
  plt.xlabel('To State')
  plt.ylabel('From State')
  plt.tight_layout()
  plt.show()

plot_transition_matrix(model.A, STATE_NAMES)

## 9. Visualize Emission Probabilities (State Means)

In [ ]:
def plot_emission_means(means, feature_names, state_names):
  """I plot the mean feature values for each hidden state."""
  n_states = len(state_names)
  
  fig, axes = plt.subplots(1, n_states, figsize=(16, 4))
  
  for s in range(n_states):
    axes[s].bar(range(len(feature_names)), means[s])
    axes[s].set_title(f'State: {state_names[s]}')
    axes[s].set_xticks(range(len(feature_names)))
    axes[s].set_xticklabels(feature_names, rotation=45, ha='right')
    axes[s].grid(True, alpha=0.3)
  
  plt.suptitle('Emission Means per Hidden State (after normalization)')
  plt.tight_layout()
  plt.show()

plot_emission_means(model.means, FEATURE_NAMES, STATE_NAMES)

## 10. Decode Activity Sequences with Viterbi

In [ ]:
def plot_decoded_sequence(observations, true_labels, predicted_states, title):
  """I plot the true vs predicted activity states over time."""
  T = len(observations)
  time_steps = np.arange(T)
  
  fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
  
  axes[0].step(time_steps, true_labels, where='mid', label='True', color='green')
  axes[0].set_ylabel('State')
  axes[0].set_title(f'{title} - True Activity')
  axes[0].set_yticks(range(4))
  axes[0].set_yticklabels([STATE_NAMES[i] for i in range(4)])
  axes[0].grid(True, alpha=0.3)
  
  axes[1].step(time_steps, predicted_states, where='mid', label='Predicted', color='blue')
  axes[1].set_ylabel('State')
  axes[1].set_xlabel('Window Index')
  axes[1].set_title(f'{title} - Predicted Activity (Viterbi)')
  axes[1].set_yticks(range(4))
  axes[1].set_yticklabels([STATE_NAMES[i] for i in range(4)])
  axes[1].grid(True, alpha=0.3)
  
  plt.tight_layout()
  plt.show()


# I show decoded sequences for a few test recordings
for i in range(min(4, len(test_sequences_scaled))):
  predicted = model.viterbi(test_sequences_scaled[i])
  true = test_labels[i]
  activity_name = STATE_NAMES[true[0]]
  plot_decoded_sequence(test_sequences_scaled[i], true, predicted, f'Test Sample: {activity_name}')

## 11. Evaluate on Unseen Test Data

In [ ]:
def evaluate_model(model, test_sequences, test_labels):
  """I evaluate the model and compute sensitivity, specificity, and accuracy."""
  all_true = []
  all_pred = []
  
  for seq, labels in zip(test_sequences, test_labels):
    predicted = model.viterbi(seq)
    all_true.extend(labels)
    all_pred.extend(predicted)
  
  all_true = np.array(all_true)
  all_pred = np.array(all_pred)
  
  return all_true, all_pred


y_true, y_pred = evaluate_model(model, test_sequences_scaled, test_labels)
overall_accuracy = accuracy_score(y_true, y_pred)
print(f"Overall Accuracy: {overall_accuracy:.4f}")

In [ ]:
def compute_per_class_metrics(y_true, y_pred, n_classes):
  """I compute sensitivity and specificity for each activity class."""
  results = []
  
  for s in range(n_classes):
    # True positives: correctly predicted as class s
    tp = np.sum((y_true == s) & (y_pred == s))
    # False negatives: actually class s but predicted as something else
    fn = np.sum((y_true == s) & (y_pred != s))
    # False positives: predicted as class s but actually something else
    fp = np.sum((y_true != s) & (y_pred == s))
    # True negatives: correctly predicted as not class s
    tn = np.sum((y_true != s) & (y_pred != s))
    
    # Sensitivity = TP / (TP + FN)
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    # Specificity = TN / (TN + FP)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    # Class accuracy
    n_samples = tp + fn
    class_accuracy = tp / n_samples if n_samples > 0 else 0
    
    results.append({
      'State': STATE_NAMES[s],
      'Number of Samples': n_samples,
      'Sensitivity': sensitivity,
      'Specificity': specificity,
      'Overall Accuracy': class_accuracy
    })
  
  return pd.DataFrame(results)


metrics_df = compute_per_class_metrics(y_true, y_pred, n_states)
print("\nEvaluation Results on Unseen Test Data:")
print(metrics_df.to_string(index=False))
print(f"\nOverall Accuracy: {overall_accuracy:.4f}")

In [ ]:
def plot_confusion_matrix(y_true, y_pred, state_names):
  """I plot a confusion matrix for the test results."""
  cm = confusion_matrix(y_true, y_pred)
  labels = [state_names[i] for i in range(len(state_names))]
  
  plt.figure(figsize=(8, 6))
  sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
              xticklabels=labels, yticklabels=labels)
  plt.title('Confusion Matrix (Unseen Test Data)')
  plt.xlabel('Predicted Activity')
  plt.ylabel('True Activity')
  plt.tight_layout()
  plt.show()

plot_confusion_matrix(y_true, y_pred, STATE_NAMES)

## 12. Analysis and Reflection

**Easiest vs Hardest Activities to Distinguish:**
- Still is usually the easiest because it has very low motion and variance.
- Jumping is also fairly easy due to high acceleration spikes.
- Standing and walking can be harder to separate because both involve relatively steady motion at waist level.

**Transition Probabilities:**
- The diagonal values in the transition matrix are high, meaning the model expects to stay in the same activity.
- This makes sense because each recording contains only one activity.

**Sensor Noise and Sampling Rate:**
- My data was collected at ~100 Hz which is good for capturing human motion.
- At this rate, a 1-second window gives 100 samples which is enough for stable features and FFT.
- Noise can affect variance and spectral energy features, which is why I normalized with Z-score.

**Possible Improvements:**
- Collect more data (more samples per activity)
- Record combined activity sequences (e.g., stand then walk then jump)
- Add more features like RMS or peak frequency
- Use a phone strap to keep sensor position consistent